# dfthcsos on a GPU: lookup-aware rounding, kernel-search loaders, official verifier

Rounds marculera's continuous (8, 143, 45) compact-group table to 8-bit words with a **lookup-aware code ladder**: more
batches with bit 0 as a Reed–Muller RM(1,4) codeword per 16-address block. Each extra coded circuit saves 22 Toffoli
products per pass. The run then refines and refits the weights, compiles the 15 ANGLE circuits with
qwertykeypadapple's kernel-relation search on Lera's tensor-lookup compiler, and scores with the **unmodified** official
`dfthcsos` verifier.

**Checkpointing.** The rounding writes a checkpoint after every stage to `/kaggle/working/ckpt/`, and stops cleanly
when `BUDGET_H` is used up. To resume, attach this notebook's previous version output as an input and run again. The
checkpoint is found automatically.

**Credits (Apache-2.0).**
- **Lera (marculera):** the DF-THC compact-groups pipeline `dfthc-compact-pipeline`, i.e. the seed, Adam table,
  progressive rounding `quantize_rm.py`, `rmcode`, `refine`, `refit_al` and `tensorqrom`. Lera in turn credits
  ahmedberatozer, somejoint, mrmeows, heliosli, lightningv08, kozykappa, Kozykorpesh, Sumit Chongder, anazemcev,
  phucthaiv02, Michael Rustad, k256net and Dmitrii Gluzdov.
- **qwertykeypadapple:** the kernel-relation search (`qcc-kernel-search-portfolio`).
- **Google Quantum AI:** the starter kit `dfthc` and the verifier.

Our changes: stage checkpoint and resume for `quantize_rm.py` (a patch applied below; the algorithm is unchanged), the
code ladder, and the orchestration.

In [ ]:
# DATASETS: marculera/dfthc-compact-pipeline, qwertykeypadapple/qcc-kernel-search-portfolio, sohier/verifiable-circuit-compilation-starter-kit
# COMPETITION: google-verifiable-quantum-circuit-compilation
# ---- configuration ----------------------------------------------------------------------------------------------
TABLE      = 'adam_8x143x45.npz'   # marculera's continuous table (error_2 14.155 mHa before rounding)
CODES      = '0:1:2-14'            # lookup-aware ladder: '0:1:3-14' (Lera, 19.95 mHa), '0:1:2-14' (-44 T), '0:1:0-14' (-176 T, likely > 20 mHa)
LBFGS_IT   = 60000                 # L-BFGS iterations per rounding stage (Lera's converged schedule; 30k did not pass)
BUDGET_H   = 8.4                   # wall-clock hours for the rounding in this session, then checkpoint and stop
KAPPA      = 2e-5
REFIT_TARGET_MHA = 19.995          # refit_al target for error_2; keeps ~5e-6 Ha margin on the 20 mHa gate
SEARCH_SEEDS = '0:4'               # kernel-search seeds per circuit, first pass (CPU, parallel)
SEARCH_SEEDS2 = '0:2'              # second, wider pass (layouts sixes+fives, trials 1 and 6)
SCHEDULE = ('0.9995,0.9990,0.9980,0.9970,0.9960,0.9950,0.9930,0.9910,0.9890,0.9860,0.9830,0.9800,0.9750,0.9700,'
            '0.9650,0.9600,0.9550,0.9500,0.9400,0.9300,0.9200,0.9100,0.9000,0.8800,0.8600,0.8400,0.8200,0.7950,'
            '0.7700,0.7450,0.7200,0.6900,0.6600,0.6300,0.6000,0.5700,0.5400,0.5100,0.4800,0.4500,0.4200,0.3900,'
            '0.3600,0.3300,0.3000,0.2750,0.2500,0.2250,0.2000,0.1600,0.1200,0.0900,0.0650,0.0450,0.0300,0.0200,'
            '0.0120,0.0060,0.0000')   # the 59-stage schedule of Lera's round_845_60k run

In [ ]:
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path
T_START = time.time()
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
def find1(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return Path(hits[0])
COMP = next(Path(p) for p in ['/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation',
                              '/kaggle/input/google-verifiable-quantum-circuit-compilation'] if Path(p, 'verifiers').is_dir())
PIPE = find1('dfthc_refit/fastfit.py').parent
QK = find1('src/kernel_search.py').parent
STARTER = find1('dfthc/df_thc_sga.py').parents[1]
print('competition', COMP, '| pipeline', PIPE, '| kernel search', QK, '| starter kit', STARTER)
# the layout the kernel-search drivers expect: P/scripts/{kernel_search,...}.py and P/scripts/marculera/dfthc_refit
P = WORK / 'qk'; SCR = P / 'scripts'
if SCR.exists(): shutil.rmtree(SCR)
(SCR / 'marculera').mkdir(parents=True)
shutil.copytree(PIPE, SCR / 'marculera' / 'dfthc_refit')
for f in QK.glob('*.py'): shutil.copy(f, SCR / f.name)
E = SCR / 'marculera' / 'dfthc_refit' / 'experiments'
os.environ.update(DFTHC_DATA=str(COMP), DFTHC_STARTER=str(STARTER), PYTHONUNBUFFERED='1')
try:
    import optax  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optax'], check=True)
try:
    import kickmix  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kickmix'], check=False)
subprocess.run('nvidia-smi || echo no GPU visible', shell=True)
try:
    import jax; print('jax devices:', jax.devices())
except Exception as e:
    print('jax import failed:', e)

In [ ]:
import base64
PATCH_B64 = 'IiIiUGF0Y2ggbWFyY3VsZXJhJ3MgZXhwZXJpbWVudHMvcXVhbnRpemVfcm0ucHkgKEFwYWNoZS0yLjApIHdpdGggc3RhZ2UgY2hlY2twb2ludHMsIHJlc3VtZSBhbmQgYSB3YWxsLWNsb2NrIGJ1ZGdldC4KCiAgICBweXRob24gcGF0Y2hfcXVhbnRpemVfcmVzdW1lLnB5IFNSQy9xdWFudGl6ZV9ybS5weSBEU1QvcXVhbnRpemVfcm1fY2twdC5weQoKVGhlIHBhdGNoZWQgc2NyaXB0IHJlYWRzIHR3byBlbnZpcm9ubWVudCB2YXJpYWJsZXM6CiAgUVJNX0NLUFQgICAgcGF0aCBvZiB0aGUgY2hlY2twb2ludCAubnB6ICh3cml0dGVuIGFmdGVyIGV2ZXJ5IHN0YWdlLCByZWFkIGF0IHN0YXJ0IGlmIHByZXNlbnQpCiAgUVJNX0JVREdFVCAgc2Vjb25kcyBvZiB3YWxsIGNsb2NrIGFmdGVyIHdoaWNoIGl0IHNhdmVzIGFuZCBleGl0cyB3aXRoIHN0YXR1cyAzIChyZXN1bWUgaW4gYSBsYXRlciBzZXNzaW9uKQpOb3RoaW5nIGVsc2UgaW4gdGhlIGFsZ29yaXRobSBjaGFuZ2VzLiBNb2RpZmljYXRpb24gbm90aWNlIChBcGFjaGUtMi4wIHNlYy4gNGIpOiBjaGVja3BvaW50aW5nIGFkZGVkIGJ5IEFnZW50IEMuCiIiIgppbXBvcnQgc3lzCgpzcmMsIGRzdCA9IHN5cy5hcmd2WzFdLCBzeXMuYXJndlsyXQpzID0gb3BlbihzcmMpLnJlYWQoKQoKYW5jaG9yID0gIiAgICBzY2hlZHVsZSA9IHR1cGxlKGZsb2F0KHgpIGZvciB4IGluIGEuZnJlZS5zcGxpdCgnLCcpKSBpZiBhLmZyZWUgZWxzZSBGUkVFXG4iCmFzc2VydCBhbmNob3IgaW4gcwpja3B0X2NvZGUgPSBhbmNob3IgKyAnJycgICAgaW1wb3J0IG9zIGFzIF9vcwogICAgX0NLID0gX29zLmVudmlyb24uZ2V0KCdRUk1fQ0tQVCcpOyBfQlVER0VUID0gZmxvYXQoX29zLmVudmlyb24uZ2V0KCdRUk1fQlVER0VUJywgJzFlMTgnKSk7IF9UMCA9IHRpbWUudGltZSgpCiAgICBzdGFydF9zdGFnZSA9IDAKICAgIGlmIF9DSyBhbmQgX29zLnBhdGguZXhpc3RzKF9DSyk6CiAgICAgICAgd2l0aCBucC5sb2FkKF9DSywgYWxsb3dfcGlja2xlPUZhbHNlKSBhcyBfejoKICAgICAgICAgICAgdGhldGEsIHcgPSBucC5hcnJheShfelsndGhldGEnXSksIGYzMihucC5hcnJheShfelsndyddKSkKICAgICAgICAgICAgZnJvemVuLCBxdmFscyA9IG5wLmFycmF5KF96Wydmcm96ZW4nXSksIG5wLmFycmF5KF96WydxdmFscyddKQogICAgICAgICAgICBrZXkgPSBqYXgucmFuZG9tLndyYXBfa2V5X2RhdGEoam5wLmFzYXJyYXkoX3pbJ2tleSddKSkKICAgICAgICAgICAgcmVzZXJ2ZWQgPSBucC5hcnJheShfelsncmVzZXJ2ZWQnXSkgaWYgYm9vbChfelsnaGFzX3Jlc2VydmVkJ10pIGVsc2UgTm9uZQogICAgICAgICAgICBsb2NrZWQgPSAobnAuYXJyYXkoX3pbJ2xvY2tlZF9yJ10pLCBucC5hcnJheShfelsnbG9ja2VkX20nXSkpIGlmIGJvb2woX3pbJ2hhc19sb2NrZWQnXSkgZWxzZSBOb25lCiAgICAgICAgICAgIGNsaXBwaW5nID0gYm9vbChfelsnY2xpcHBpbmcnXSk7IHN0YXJ0X3N0YWdlID0gaW50KF96WydzdGFnZSddKQogICAgICAgIHByaW50KGYncmVzdW1lZCBmcm9tIHtfQ0t9IGF0IHN0YWdlIHtzdGFydF9zdGFnZX0gKGZyb3plbiB7ZnJvemVuLm1lYW4oKTouNGZ9KScsIGZsdXNoPVRydWUpCgogICAgZGVmIF9ja3B0KG5leHRfc3RhZ2UpOgogICAgICAgIGlmIG5vdCBfQ0s6CiAgICAgICAgICAgIHJldHVybgogICAgICAgIHRtcCA9IF9DSyArICcudG1wLm5weicKICAgICAgICBucC5zYXZleih0bXAsIHRoZXRhPW5wLmFzYXJyYXkodGhldGEpLCB3PW5wLmFzYXJyYXkodyksIGZyb3plbj1mcm96ZW4sIHF2YWxzPXF2YWxzLAogICAgICAgICAgICAgICAgIGtleT1ucC5hc2FycmF5KGpheC5yYW5kb20ua2V5X2RhdGEoa2V5KSksIGhhc19yZXNlcnZlZD1yZXNlcnZlZCBpcyBub3QgTm9uZSwKICAgICAgICAgICAgICAgICByZXNlcnZlZD1yZXNlcnZlZCBpZiByZXNlcnZlZCBpcyBub3QgTm9uZSBlbHNlIG5wLnplcm9zKDEsIGJvb2wpLCBoYXNfbG9ja2VkPWxvY2tlZCBpcyBub3QgTm9uZSwKICAgICAgICAgICAgICAgICBsb2NrZWRfcj1sb2NrZWRbMF0gaWYgbG9ja2VkIGlzIG5vdCBOb25lIGVsc2UgbnAuemVyb3MoMSksIGxvY2tlZF9tPWxvY2tlZFsxXSBpZiBsb2NrZWQgaXMgbm90IE5vbmUgZWxzZSBucC56ZXJvcygxKSwKICAgICAgICAgICAgICAgICBjbGlwcGluZz1jbGlwcGluZywgc3RhZ2U9bmV4dF9zdGFnZSkKICAgICAgICBfb3MucmVwbGFjZSh0bXAsIF9DSykKICAgICAgICBpZiB0aW1lLnRpbWUoKSAtIF9UMCA+IF9CVURHRVQgYW5kIG5leHRfc3RhZ2UgPCBsZW4oc2NoZWR1bGUpICsgMToKICAgICAgICAgICAgcHJpbnQoZididWRnZXQgcmVhY2hlZCBhZnRlciBzdGFnZSB7bmV4dF9zdGFnZSAtIDF9OiBjaGVja3BvaW50IHNhdmVkLCBleGl0aW5nIGZvciByZXN1bWUnLCBmbHVzaD1UcnVlKQogICAgICAgICAgICBzeXMuZXhpdCgzKQonJycKcyA9IHMucmVwbGFjZShhbmNob3IsIGNrcHRfY29kZSkKCmxvb3AgPSAiICAgIGZvciBzdGFnZSwgZnJlZSBpbiBlbnVtZXJhdGUoc2NoZWR1bGUgKyAoTm9uZSwpKTpcbiIKYXNzZXJ0IGxvb3AgaW4gcwpzID0gcy5yZXBsYWNlKGxvb3AsIGxvb3AgKyAiICAgICAgICBpZiBzdGFnZSA8IHN0YXJ0X3N0YWdlOlxuICAgICAgICAgICAgY29udGludWVcbiIpCgpsYiA9ICIgICAgICAgICAgICAgICAgICBmJ0Qge2Zsb2F0KGRfcGVuYWx0eSh3KSk6Ny4yZn0gIEwtQkZHUyAoe3RpbWUudGltZSgpIC0gdDA6LjBmfSBzKScsIGZsdXNoPVRydWUpXG4gICAgICAgICAgICBjb250aW51ZVxuIgphc3NlcnQgbGIgaW4gcwpzID0gcy5yZXBsYWNlKGxiLCBsYi5yZXBsYWNlKCIgICAgICAgICAgICBjb250aW51ZVxuIiwgIiAgICAgICAgICAgIF9ja3B0KHN0YWdlICsgMSlcbiAgICAgICAgICAgIGNvbnRpbnVlXG4iKSkKCmFkID0gIiAgICAgICAgICAgICAgZidEIHtmbG9hdChkX3BlbmFsdHkodykpOjcuMmZ9ICAoe3RpbWUudGltZSgpIC0gdDA6LjBmfSBzKScsIGZsdXNoPVRydWUpXG4iCmFzc2VydCBhZCBpbiBzCnMgPSBzLnJlcGxhY2UoYWQsIGFkICsgIiAgICAgICAgX2NrcHQoc3RhZ2UgKyAxKVxuIikKCiMgYHJlc2VydmVkYCwgYGxvY2tlZGAgYW5kIGBjbGlwcGluZ2AgYXJlIGFzc2lnbmVkIGluIG1haW4gYW5kIHJlYWQgYnkgbmVzdGVkIGZ1bmN0aW9uczsgdGhlIHJlc3VtZSBibG9jayBhc3NpZ25zIHRoZW0KIyBpbiBtYWluJ3Mgc2NvcGUsIHdoaWNoIGlzIHdoZXJlIHRoZXkgYWxyZWFkeSBsaXZlLCBzbyBubyBub25sb2NhbCBjaGFuZ2VzIGFyZSBuZWVkZWQuCm9wZW4oZHN0LCAndycpLndyaXRlKHMpCnByaW50KCdwYXRjaGVkIC0+JywgZHN0KQo='
Path(WORK / 'patch_quantize_resume.py').write_bytes(base64.b64decode(PATCH_B64))
subprocess.run([sys.executable, str(WORK / 'patch_quantize_resume.py'), str(E / 'quantize_rm.py'), str(E / 'quantize_rm_ckpt.py')], check=True)

In [ ]:
# ---- stage 1: lookup-aware progressive rounding (GPU), resumable --------------------------------------------------
CK = WORK / 'ckpt'; CK.mkdir(exist_ok=True)
ROUNDED = WORK / 'rounded.npz'
prev = sorted(glob.glob('/kaggle/input/**/ckpt/qrm.npz', recursive=True)) + sorted(glob.glob('/kaggle/input/**/rounded.npz', recursive=True))
for p in prev:                      # resume from an attached previous version of this notebook
    dst = ROUNDED if p.endswith('rounded.npz') else CK / 'qrm.npz'
    if not dst.exists():
        shutil.copy(p, dst); print('resuming from', p)
status = 'done' if ROUNDED.exists() else None
if status is None:
    env = dict(os.environ, QRM_CKPT=str(CK / 'qrm.npz'), QRM_BUDGET=str(BUDGET_H * 3600 - (time.time() - T_START)))
    cmd = [sys.executable, '-u', str(E / 'quantize_rm_ckpt.py'), str(find1(TABLE)), str(ROUNDED), '--brot', '8', '--open',
           '--codes', CODES, '--free', SCHEDULE, '--lbfgs-until', '1.0', '--lbfgs-iters', str(LBFGS_IT), '--kappa', str(KAPPA)]
    with open(WORK / 'round.log', 'a') as log:
        rc = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, env=env).returncode
    print(open(WORK / 'round.log').read()[-3000:])
    status = 'done' if rc == 0 else 'paused' if rc == 3 else f'failed ({rc})'
print('rounding:', status)
(WORK / 'status.json').write_text(json.dumps({'rounding': status, 'codes': CODES, 'lbfgs_iters': LBFGS_IT}))

In [ ]:
# ---- stage 2: refine the one-body rows and refit the weights to the gates (CPU) ---------------------------------
if status == 'done' and not (WORK / 'finish' / 'refit_al.npz').exists():
    cmd = [sys.executable, '-u', str(E / 'finish_rm.py'), str(ROUNDED), str(WORK / 'finish'), '--target', str(REFIT_TARGET_MHA),
           '--trials', '6', '--shots', '64']
    with open(WORK / 'finish.log', 'w') as log:
        subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, env=dict(os.environ, OPENBLAS_NUM_THREADS='1'))
    print(open(WORK / 'finish.log').read()[-2500:])      # includes Lera's own build + official verify as a baseline

In [ ]:
# ---- stage 3: kernel-relation search loaders (CPU, parallel) + build + official verifier -------------------------
TABLE_FINAL = WORK / 'finish' / 'refit_al.npz'
if status == 'done' and TABLE_FINAL.exists():
    plans, cand = WORK / 'plans', WORK / 'candidate' / 'dfthcsos'
    jobs = str(max(1, os.cpu_count()))
    for seeds, args in ((SEARCH_SEEDS, ['--engine', 'kernel', '--cache', 'full', '--layouts', 'sixes', '--margins', '0', '--trials', '1']),
                        (SEARCH_SEEDS2, ['--engine', 'kernel', '--cache', 'full', '--layouts', 'sixes,fives', '--margins', '0', '--trials', '1,6'])):
        subprocess.run([sys.executable, '-u', str(SCR / 'marculera_plan_search.py'), str(TABLE_FINAL), str(plans),
                        '--seeds', seeds, '--jobs', jobs, *args], env=dict(os.environ, OPENBLAS_NUM_THREADS='1'))
    if cand.exists(): shutil.rmtree(cand)
    subprocess.run([sys.executable, '-u', str(SCR / 'marculera_plan_build.py'), str(TABLE_FINAL), str(plans), str(cand)], check=True)

In [ ]:
# ---- stage 4: the unmodified official verifier on the candidate folder, default and a private salt ----------------
sys.path.insert(0, str(COMP / 'verifiers'))
from dfthcsos import verify as V
REF = 3355599588.3155375
cand = WORK / 'candidate' / 'dfthcsos'
if cand.exists():
    (cand / 'verify.json').unlink(missing_ok=True)        # the candidate folder holds only angle_k_*.kmb + submission.npz
    out = {}
    for salt in ('', 'agentC-private-check'):
        r = V.verify(str(cand), salt=salt)
        out[salt or 'default'] = {k: v for k, v in r.items() if isinstance(v, (int, float, str, bool))} | {
            'ratio': REF / float(r['score']), 'metrics': r['metrics']}
        print(f"salt {salt!r}: {r['status']} lambda_eff {r['lambda_eff']:.6f} T {r['toffoli_count']:.0f} Q {r['num_qubits']} "
              f"e1 {1e3*r['metrics']['error_1']:.6f} e2 {1e3*r['metrics']['error_2']:.6f} mHa E_gap {r['metrics']['E_gap']:.9f} "
              f"ratio {REF / float(r['score']):.4f}")
    (WORK / 'candidate_verify.json').write_text(json.dumps(out, indent=1, default=str))
    shutil.make_archive(str(WORK / 'candidate_dfthcsos'), 'zip', cand)
print('elapsed', round((time.time() - T_START) / 3600, 2), 'h')